# 01 — xarray Basics with ERA5 WeatherBench Data

This notebook explores the WeatherBench 5.625° dataset using xarray.
We cover: loading NetCDF, inspecting dimensions/coords, slicing, plotting, and Dask.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load the dataset
First, download data with: `uv run python -m weather_ml.data.download`

In [ ]:
ds = xr.open_mfdataset('../data/*.nc', combine='by_coords')
ds

## 2. Inspect dimensions and coordinates

In [ ]:
print('Dimensions:', dict(ds.dims))
print('Variables:', list(ds.data_vars))
print('Coordinates:', list(ds.coords))
print()
print('Lat range:', ds.lat.values.min(), 'to', ds.lat.values.max())
print('Lon range:', ds.lon.values.min(), 'to', ds.lon.values.max())
print('Time range:', ds.time.values[0], 'to', ds.time.values[-1])
print('Time steps:', len(ds.time))

## 3. Access a DataArray

In [ ]:
z500 = ds['z']  # geopotential at 500 hPa
print(z500)
print()
print('Shape:', z500.shape)
print('Dtype:', z500.dtype)

## 4. Selection: `.sel()` and `.isel()`

In [ ]:
# Select by label — a single time step
snapshot = z500.sel(time='2017-01-01T00:00')
print('sel by label:', snapshot.shape)

# Select by index
snapshot_idx = z500.isel(time=0)
print('isel by index:', snapshot_idx.shape)

# Slice a date range
jan_2017 = z500.sel(time=slice('2017-01-01', '2017-01-31'))
print('Jan 2017 slice:', jan_2017.shape)

## 5. Plot a global map

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

z500.sel(time='2017-01-01T00:00').plot(ax=axes[0], cmap='viridis')
axes[0].set_title('Geopotential 500 hPa — 2017-01-01 00Z')

ds['t'].sel(time='2017-01-01T00:00').plot(ax=axes[1], cmap='RdYlBu_r')
axes[1].set_title('Temperature 850 hPa — 2017-01-01 00Z')

plt.tight_layout()
plt.show()

## 6. Time series at a single location

In [ ]:
# Paris: lat≈48.9, lon≈2.3
paris_t850 = ds['t'].sel(lat=48.9, lon=2.3, method='nearest')
paris_2017 = paris_t850.sel(time=slice('2017-01-01', '2017-12-31'))

paris_2017.plot(figsize=(14, 3))
plt.title('Temperature 850 hPa near Paris — 2017')
plt.ylabel('Temperature (K)')
plt.show()

## 7. groupby / resample

In [ ]:
# Monthly mean of geopotential
z_monthly = z500.sel(time=slice('2017', '2017')).resample(time='1ME').mean()
print('Monthly means shape:', z_monthly.shape)

# Seasonal climatology
z_seasonal = z500.groupby('time.season').mean(dim='time')
print('Seasonal climatology shape:', z_seasonal.shape)

In [ ]:
# Plot seasonal climatology
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, season in zip(axes.flat, ['DJF', 'MAM', 'JJA', 'SON']):
    z_seasonal.sel(season=season).plot(ax=ax, cmap='viridis')
    ax.set_title(f'Z500 — {season}')
plt.tight_layout()
plt.show()

## 8. Dask chunking for lazy computation

In [ ]:
# Lazy loading with Dask chunks
ds_lazy = xr.open_mfdataset('../data/*.nc', combine='by_coords', chunks={'time': 100})
print('Lazy dataset:')  
print(ds_lazy['z'])
print()
print('Chunk sizes:', ds_lazy['z'].chunks)

In [ ]:
# Computation is lazy until .compute() or .values
mean_lazy = ds_lazy['z'].mean(dim='time')
print('Result type:', type(mean_lazy.data))  # dask array

# Trigger computation
mean_result = mean_lazy.compute()
print('Computed shape:', mean_result.shape)

## Summary

| Operation | Code |
|-----------|------|
| Load | `xr.open_dataset('file.nc')` |
| Multi-file | `xr.open_mfdataset('*.nc')` |
| Label select | `ds.sel(time='2017-01-01')` |
| Index select | `ds.isel(time=0)` |
| Interpolate | `ds.interp(lat=48.5)` |
| Resample | `ds.resample(time='1ME').mean()` |
| Group | `ds.groupby('time.season').mean()` |
| Lazy/Dask | `xr.open_mfdataset(..., chunks={'time': 100})` |